# DT401 Week 2 Section 4: Exploring Data Architecture with SQL

**Module:** Data Fundamentals  
**Week:** 2 of 7  
**Section:** 4 of 4  
**Estimated Time:** 30–40 minutes

---

## Learning Objectives

By the end of this notebook, you will be able to:

- Interpret an Entity Relationship Diagram showing tables, attributes, and relationships
- Explain how normalisation reduces redundancy by observing a worked example progressing from unnormalised to 3NF
- Read and modify basic SQL queries (SELECT, WHERE, ORDER BY) to retrieve specific data from a database
- Connect data architecture concepts to a working database you can interact with

---

## Context Recap

In Section 3, you explored how the same data can be represented as CSV, JSON, and SQL tables. This notebook focuses on the SQL side — how databases are designed, structured, and queried. You will see the architecture concepts from Section 4 (ERDs, normalisation, SQL commands) brought to life in a working database you can interact with.

---

## A Note Before You Start

This notebook is about **exploring** data architecture — not writing code from scratch.

Every code cell in this notebook has already been written for you. Your job is to **run** each cell, **read** the output, and **think** about what it means in the context of the learning you have just completed.

This section introduces slightly more SQL interaction than Section 3. Some tasks will ask you to **modify a single line** in a SQL query — always with explicit instructions about exactly what to change. One stretch task provides a partial query for you to complete. You are not expected to write SQL from scratch; that is what Workshop 1 is for.

You are building **familiarity** with SQL, not fluency. Fluency comes with practice in the workshop and in later modules.

---

## How to Use This Notebook

You will find three types of box throughout this notebook:

<div class="alert alert-block alert-warning">
<b>Reminder boxes:</b><br>
Important context about how this notebook connects to the other material you are studying or to your Module Challenge.
</div>

<div class="alert alert-block alert-info">
<b>What just happened?</b><br>
Explanations of what a code cell did and what you should take from the output.
</div>

<div class="alert alert-block alert-success">
<b>Your turn:</b><br>
Tasks for you to complete. Some involve editing a single value in a SQL cell. Others ask you to fill in a table in a markdown cell. Instructions will always be explicit — you will never be asked to write code from scratch.

Tasks may include a hidden solution.
<details>
    <summary style="color:black;font-weight:bold">Click here to reveal the solution</summary>

Solutions appear here.

</details>
</div>

---

## Setup

Run the cell below to load the libraries this notebook uses and connect to a temporary database. You should see a confirmation message. The database lives entirely in memory — it resets each time you restart the kernel.

In [ ]:
import pandas as pd
import json
from IPython.display import display
%config SqlMagic.displaylimit = 0
%config SqlMagic.autopandas = True
%load_ext sql
%sql sqlite:///:memory:

print("Setup complete — database ready.")
print("This notebook uses an in-memory SQLite database.")
print("All tables are created fresh each time you run the notebook from the top.")

---

## Part 1: From Concept to Database — Designing a Data Model

### A Workplace Scenario

In Section 3, you saw clinic appointment data stored as CSV and JSON — a flat record for each appointment, with a patient ID, date, and whether the patient attended. That example kept things simple on purpose. A real GP practice needs something more structured.

Consider what a GP practice actually needs to track:

- **Patients** — name, date of birth, NHS number
- **Doctors** — name, specialty
- **Appointments** — date, time, appointment type, which patient, which doctor, whether the patient attended

These three categories of information are related to one another. An appointment is meaningless without a patient and a doctor. This is the essence of a **relational database**: information is split into separate tables, and the relationships between them are recorded explicitly.

This was described when it introduced data modelling — moving from a conceptual model (what do we need to track?) through a logical model (what tables and relationships do we need?) to a physical model (how do we implement it in SQL?). You are about to see all three stages applied to this scenario.

The same design principles apply whether you are building a patient records system, a retail inventory database, a council service request tracker, or a customer CRM. The scenario is a GP practice — the thinking is universal.

### The Entity Relationship Diagram

Before any SQL is written, a database architect draws an **Entity Relationship Diagram (ERD)**. This shows the tables (entities), their columns (attributes), and how the tables connect to one another (relationships and cardinality).

The diagram below shows the ERD for the GP practice database. Read it before running any code — it is the blueprint you are about to implement.

```mermaid
erDiagram
    PATIENTS {
        int patient_id PK
        text patient_name
        text date_of_birth
        text nhs_number
    }

    DOCTORS {
        int doctor_id PK
        text doctor_name
        text specialty
    }

    APPOINTMENTS {
        int appointment_id PK
        text appointment_date
        text appointment_time
        text appointment_type
        int patient_id FK
        int doctor_id FK
        boolean attended
    }

    PATIENTS ||--o{ APPOINTMENTS : "books"
    DOCTORS ||--o{ APPOINTMENTS : "conducts"
```

**What this diagram shows:** Three entities — PATIENTS, DOCTORS, and APPOINTMENTS — each with their attributes listed. The lines connecting the tables show relationships: 

- a single patient can have many appointments `||--o{` (one-to-many)
- a single doctor can conduct many appointments `||--o{` (one-to-many). 

The `||--o{` notation is Mermaid's way of drawing the relationship — `||` means exactly one, and `o{` means zero or more. 

The `patient_id` and `doctor_id` columns in APPOINTMENTS are foreign keys (FK) — they reference the primary keys (PK) in the other tables, creating the links between them.

<div class="alert alert-block alert-info">
<b>Further Information</b><br>

[A Guide to the Entity Relationship Diagram (ERD) — Database Star (Brumm, patients., 2019)](https://www.databasestar.com/entity-relationship-diagram/) — A practitioner-authored guide to ERD notation, entity types, and relationship cardinality. The concepts are foundational and stable.

*Read time: 5 minutes*
</div>

### Worked Example: Creating the Database

We will now implement the ERD above in SQL. The code cells below contain `CREATE TABLE` statements that define the three tables, their columns, data types, and relationships. Run each cell and read the output to understand how the ERD translates into SQL.

In [ ]:
%%sql
-- Create the patients table — this is the physical data model for the PATIENTS entity in the ERD
CREATE TABLE patients (
    patient_id    INTEGER PRIMARY KEY,
    patient_name  TEXT    NOT NULL,
    date_of_birth TEXT    NOT NULL,
    nhs_number    TEXT    UNIQUE NOT NULL
);

In [ ]:
%%sql
-- Create the doctors table — the DOCTORS entity from the ERD
CREATE TABLE doctors (
    doctor_id   INTEGER PRIMARY KEY,
    doctor_name TEXT    NOT NULL,
    specialty   TEXT    NOT NULL
);

In [ ]:
%%sql
-- Create the appointments table — the APPOINTMENTS entity from the ERD
-- Notice the FOREIGN KEY constraints: these enforce the relationships shown in the ERD
CREATE TABLE appointments (
    appointment_id   INTEGER PRIMARY KEY,
    appointment_date TEXT    NOT NULL,
    appointment_time TEXT    NOT NULL,
    appointment_type TEXT    NOT NULL,
    patient_id       INTEGER NOT NULL,
    doctor_id        INTEGER NOT NULL,
    attended         BOOLEAN NOT NULL,
    FOREIGN KEY (patient_id) REFERENCES patients(patient_id),
    FOREIGN KEY (doctor_id)  REFERENCES doctors(doctor_id)
);

<div class="alert alert-block alert-info">
<b>What just happened?</b><br>

You have just implemented the ERD in SQL. Each `CREATE TABLE` statement corresponds directly to one entity in the diagram:

- The column names match the attributes shown in the ERD boxes
- `INTEGER PRIMARY KEY` creates the unique identifier for each row — the PK in the diagram
- `NOT NULL` enforces that certain fields must always have a value — a patient must have a name and an NHS number
- `UNIQUE` on `nhs_number` prevents two patients from sharing the same NHS number
- `FOREIGN KEY` in the appointments table enforces the relationships drawn as lines in the ERD — an appointment must reference a real patient and a real doctor

This is the **physical data model**: the conceptual design (three related entities) has been translated into concrete SQL structure. The tables exist but contain no data yet — they are blueprints.
</div>

### Worked Example: Populating the Database

Now we have created the structure of the database, we need to add some data. The code cells below contain `INSERT INTO` statements that add records to each table. Run each cell and read the output to see how the tables are populated.

In [ ]:
%%sql
-- Insert patients
INSERT INTO patients VALUES
    (1, 'Priya Sharma',     '1991-03-14', '485 777 3456'),
    (2, 'Kwame Mensah',     '1978-11-02', '312 445 8821'),
    (3, 'Siobhan O''Reilly','1984-06-27', '729 113 0047'),
    (4, 'Amir Hussain',     '2001-01-15', '558 334 9912'),
    (5, 'Mei-Ling Wong',    '1967-09-08', '641 892 2235'),
    (6, 'Luca Romano',      '1995-04-23', '773 561 4408');

In [ ]:
%%sql
-- Insert doctors
INSERT INTO doctors VALUES
    (1, 'Dr Amara Okafor',  'General Practice'),
    (2, 'Dr Rajesh Kumar',  'General Practice'),
    (3, 'Dr Katarina Novak','Cardiology');

In [ ]:
%%sql
-- Insert appointments
-- patient_id and doctor_id are foreign keys — they reference the IDs in the other tables
INSERT INTO appointments VALUES
    (1,  '2026-01-06', '09:00', 'GP Consultation', 1, 1, TRUE),
    (2,  '2026-01-06', '09:15', 'Blood Test',      2, 1, TRUE),
    (3,  '2026-01-06', '10:00', 'Follow-up',       3, 2, FALSE),
    (4,  '2026-01-07', '08:45', 'GP Consultation', 4, 2, TRUE),
    (5,  '2026-01-07', '11:30', 'Cardiology Review',5, 3, TRUE),
    (6,  '2026-01-08', '09:00', 'GP Consultation', 6, 1, TRUE),
    (7,  '2026-01-08', '10:15', 'Follow-up',       1, 2, TRUE),
    (8,  '2026-01-09', '14:00', 'Blood Test',      3, 1, FALSE),
    (9,  '2026-01-09', '15:00', 'Cardiology Review',2, 3, TRUE),
    (10, '2026-01-10', '09:30', 'GP Consultation', 5, 2, TRUE);

<div class="alert alert-block alert-info">
<b>What just happened?</b><br>

The database now holds 6 patients, 3 doctors, and 10 appointments. Notice that the appointments table stores `patient_id` and `doctor_id` — not the names. The name for patient 1 (Priya Sharma) is stored once in the patients table; every appointment that references `patient_id = 1` automatically refers to her.

This is how relationships work in a relational database. The appointments table does not repeat patient or doctor information — it just records the connection. This design decision is the foundation of normalisation, which you will explore in Part 2.
</div>

<div class="alert alert-block alert-success">
<b>Your turn — ERD reading task:</b><br>

Look at the ERD diagram above. For each relationship, identify the type of relationship (the cardinality) and explain what it means in plain English.

Edit this cell directly to fill in the empty columns.

<details>
<summary style="font-weight:bold">Hint</summary>
Here are some common relationship types to look for:

- One-to-many (1:N): One record in Table A can relate to many records in Table patients, but each record in Table B relates to only one record in Table A. 
- One-to-one (1:1): Each record in Table A relates to exactly one record in Table B, and vice versa.
- Many-to-many (M:N): Records in Table A can relate to many records in Table B, and records in Table B can relate to many records in Table A. This often requires a junction table to implement in SQL. 
    
</details>
</div>

| Relationship | Cardinality | What it means |
|:-------------|:------------|:--------------|
| patients → appointments | | |
| doctors → appointments | | |

<details>
    <summary style="font-weight:bold">Click here to reveal the solution</summary>

| Relationship | Cardinality | What it means |
|:-------------|:------------|:--------------|
| patients → appointments | One-to-many (1 to 0 or more) | One patient can have zero or many appointments, but each appointment belongs to exactly one patient. A new patient who has not yet booked has zero appointments; a regular patient may have many. |
| doctors → appointments | One-to-many (1 to 0 or more) | One doctor can conduct zero or many appointments, but each appointment is conducted by exactly one doctor. A newly joined doctor with no bookings yet would have zero appointments in the system. |

</details>

---

## Part 2: Normalisation in Action

### Why Normalisation Matters

The three-table design you just created looks more complex than a single flat spreadsheet. There is a reason for that complexity — it is doing important work.

**Normalisation** is the process of organising a database to reduce redundancy and protect data integrity. You have studied the three normal forms

- 1NF: Each table cell has a single value and each record is unique.
- 2NF: Each non-key attribute in a table record is fully functionally dependent on the primary key.
- 3NF: Each non-key attribute in a table record is fully functionally dependent on the primary key and not on any other non-key attribute.

This part explores what happens when those principles are ignored, and why the normalised design is worth the extra effort.

Poorly normalised databases create real operational problems:

- A staff member updates a doctor's name in one appointment record but misses another. 
- A patient changes address, and the old address persists in half the rows. 

Data becomes inconsistent — and inconsistent data leads to incorrect reports, incorrect decisions, and, in a healthcare setting, potential patient safety risks.

### Worked Example: The Unnormalised Table

Let's look at what a database table would look like if someone simply put everything in one table. We will create a table — `appointments_flat` — containing all patient, doctor, and appointment information, with no relationships. 

In [ ]:
%%sql
-- The unnormalised version: all data in one flat table
-- Notice that patient details and doctor details repeat for every appointment
CREATE TABLE IF NOT EXISTS appointments_flat (
    appointment_id   INTEGER,
    appointment_date TEXT,
    appointment_time TEXT,
    appointment_type TEXT,
    patient_name     TEXT,
    date_of_birth    TEXT,
    nhs_number       TEXT,
    doctor_name      TEXT,
    specialty        TEXT,
    attended         BOOLEAN
);

INSERT INTO appointments_flat VALUES
    (1,  '2026-01-06','09:00','GP Consultation','Priya Sharma',    '1991-03-14','485 777 3456','Dr Amara Okafor', 'General Practice',TRUE),
    (2,  '2026-01-06','09:15','Blood Test',     'Kwame Mensah',    '1978-11-02','312 445 8821','Dr Amara Okafor', 'General Practice',TRUE),
    (3,  '2026-01-06','10:00','Follow-up',      'Siobhan O''Reilly','1984-06-27','729 113 0047','Dr Rajesh Kumar', 'General Practice',FALSE),
    (4,  '2026-01-07','08:45','GP Consultation','Amir Hussain',    '2001-01-15','558 334 9912','Dr Rajesh Kumar', 'General Practice',TRUE),
    (5,  '2026-01-07','11:30','Cardiology Review','Mei-Ling Wong', '1967-09-08','641 892 2235','Dr Katarina Novak','Cardiology',     TRUE),
    (6,  '2026-01-08','09:00','GP Consultation','Luca Romano',     '1995-04-23','773 561 4408','Dr Amara Okafor', 'General Practice',TRUE),
    (7,  '2026-01-08','10:15','Follow-up',      'Priya Sharma',    '1991-03-14','485 777 3456','Dr Rajesh Kumar', 'General Practice',TRUE),
    (8,  '2026-01-09','14:00','Blood Test',     'Siobhan O''Reilly','1984-06-27','729 113 0047','Dr Amara Okafor', 'General Practice',FALSE),
    (9,  '2026-01-09','15:00','Cardiology Review','Kwame Mensah',  '1978-11-02','312 445 8821','Dr Katarina Novak','Cardiology',     TRUE),
    (10, '2026-01-10','09:30','GP Consultation','Mei-Ling Wong',   '1967-09-08','641 892 2235','Dr Rajesh Kumar', 'General Practice',TRUE);

-- Display the entire flat table
SELECT * FROM appointments_flat;

<div class="alert alert-block alert-info">
<b>What just happened?</b><br>

Let's look at the output. Notice that `Dr Amara Okafor` appears in rows 1, 2, 6, and 8 — her name and specialty are repeated four times. `Priya Sharma`'s date of birth and NHS number appear in rows 1 and 7.

This is the redundancy problem. If Dr Okafor's name changes (perhaps she updates her title to `Dr Amara Okafor-Boateng`), every row containing her name must be updated individually. Miss one row and the database becomes inconsistent — two different names now refer to the same doctor. This is called an **update anomaly**.

What if you want to add a new doctor to the system before they have any appointments? You cannot — there is no appointments row to put them in. This is an **insertion anomaly**. What if all appointments for a particular patient are deleted? The patient's NHS number and date of birth disappear from the database entirely — a **deletion anomaly**.

These three anomalies are exactly what normalisation solves.
</div>

### Worked Example: The Normalised Version

Let's look at retrieving the same data from the three normalised tables you created in Part 1. First, each table individually — then a `JOIN` query that reconstructs the flat view on demand.

In [ ]:
%%sql
-- The doctors table: Dr Okafor's name appears exactly once
SELECT * FROM doctors;

In [ ]:
%%sql
-- The appointments table: references doctor_id, not the doctor's name
SELECT * FROM appointments;

In [ ]:
%%sql
-- A JOIN query reconstructs the full picture from the normalised tables
-- The database looks up names automatically using the foreign key references
SELECT
    a.appointment_date,
    a.appointment_time,
    a.appointment_type,
    p.patient_name,
    d.doctor_name,
    d.specialty,
    a.attended
FROM appointments a
JOIN patients p ON a.patient_id = p.patient_id
JOIN doctors  d ON a.doctor_id  = d.doctor_id
ORDER BY a.appointment_date, a.appointment_time;

<div class="alert alert-block alert-info">
<b>What just happened?</b><br>

The JOIN query produces the same readable output as the flat table — but the underlying data is stored without redundancy. Dr Okafor's name exists once, in the doctors table. The appointments table stores only the unique `doctor_id` for each doctor, which for Dr Okafor is `1`. The JOIN looks up the name when you need it.

If Dr Okafor updates her name, one row in the doctors table changes — and the change is instantly reflected in every query that references her. No inconsistency is possible.

This is **Third Normal Form (3NF)**: every column in each table depends on the primary key of that table, and on nothing else. 
</div>

### Unnormalised vs Normalised: A Visual Comparison

```mermaid
flowchart LR
    flat[appointments_flat one table all data repeated] --> normalised[three separate tables with relationships]
    
    normalised --> doctors[doctors name stored once]
    normalised --> appointments[appointments IDs only — no repetition]
    normalised --> patients[patients name stored once]
    doctors --> appointments
    patients --> appointments

    style flat fill:#ffffff,stroke:#000000,color:#000000,stroke-width:3px
    style normalised fill:#ffffff,stroke:#000000,color:#000000,stroke-width:3px
    style patients fill:#ffffff,stroke:#000000,color:#000000,stroke-width:2px
    style doctors fill:#ffffff,stroke:#000000,color:#000000,stroke-width:2px
    style appointments fill:#ffffff,stroke:#000000,color:#000000,stroke-width:2px
```

**What this diagram shows:** Moving from one flat table to three normalised tables eliminates redundancy. Each piece of information is stored exactly once — patient details in the patients table, doctor details in the doctors table, and appointment records in the appointments table. The appointments table references the other two through foreign keys, allowing the database to reconstruct the full picture on demand using JOIN.

<div class="alert alert-block alert-info">
<b>Further Information</b><br>

[Database Normalization: 1NF, 2NF, 3NF & BCNF Examples — DigitalOcean (DigitalOcean, 2025)](https://www.digitalocean.com/community/tutorials/database-normalization) — A clear, step-by-step walkthrough of each normal form with worked table examples. Use this to consolidate what you have observed in this Part.

*Read time: 10 minutes*
</div>

<div class="alert alert-block alert-success">
<b>Your turn — normalisation classification task:</b><br>

Look at the unnormalised `appointments_flat` table above. For each problem listed, explain what could go wrong and identify which normal form addresses it.

Edit this cell directly to fill in the empty columns.
</div>

| Problem type | What could go wrong? | Which normal form addresses this? |
|:-------------|:---------------------|:----------------------------------|
| A patient's date of birth is recorded incorrectly and needs to be corrected | | |
| A new doctor joins the practice but has not yet been assigned any appointments | | |
| Dr Okafor's specialty is updated in rows 1 and 2 but the change is missed in rows 6 and 8 | | |

<details>
    <summary style="font-weight:bold">Click here to reveal the solution</summary>

| Problem type | What could go wrong? | Which normal form addresses this? |
|:-------------|:---------------------|:----------------------------------|
| A patient's date of birth is recorded incorrectly and needs to be corrected | In the flat table, the same date of birth appears in multiple rows (once per appointment). Correcting it requires updating every row for that patient. Miss one row and the database holds two different dates of birth for the same person — an update anomaly. | 2NF and 3NF — patient details should depend only on patient_id, stored once in a separate patients table. |
| A new doctor joins the practice but has not yet been assigned any appointments | In the flat table, doctor information only appears within appointment rows. A doctor with no appointments cannot be recorded at all — an insertion anomaly. This matters operationally: you cannot look up the doctor's specialty or contact details until they have a booking. | 2NF — doctor details should be stored in their own table, independent of appointments. |
| Dr Okafor's specialty is updated in rows 1 and 2 but the change is missed in rows 6 and 8 | The database now contains two different specialties for the same doctor — an update anomaly. Any query that reports on specialty will return inconsistent results. This inconsistency is invisible unless you check every row manually. | 3NF — specialty is a fact about the doctor, not about the appointment. It belongs in the doctors table, stored once. |

</details>

---

## Part 3: Querying the Database with SQL

### Retrieving Data with SELECT

Creating and normalising a database is work that is important, but invisible to most users. What people interact with day to day is the **query**: a question put to the database in SQL, which responds with exactly the data requested.

When you look up a patient record, generate a report of this week's appointments, or filter a dashboard by doctor — SQL is running behind the scenes. The `SELECT` statement is the primary tool. You saw it briefly in Section 3's SQL preview; now you will explore it properly.

This Part moves through four worked examples, from the simplest possible query to one that combines tables and counts rows. After the worked examples, you will try making small modifications to the queries yourself.

### Worked Example 1: Basic SELECT

Here is the simplest possible SQL query — retrieving everything from one table.

- `SELECT *` means "select values from all columns"
- `FROM patients` specifies which table to query

In [ ]:
%%sql
SELECT * 
FROM patients;

<div class="alert alert-block alert-info">
<b>What just happened?</b><br>

`SELECT *` means 'select values from all columns'. `FROM patients` tells the database which table to look in. This is the simplest possible query: no filtering, no sorting, no joining. You get back exactly what is in the table.

In practice, `SELECT *` on a large table is rarely a good idea — it can return a very large table. But it is the right starting point when you are exploring a table for the first time.
</div>

### Worked Example 2: Filtering with WHERE

**What you will see:** A query that retrieves specific columns and filters to a single patient.

In [ ]:
%%sql
SELECT patient_name, date_of_birth
FROM patients
WHERE patient_name = 'Priya Sharma';

<div class="alert alert-block alert-info">
<b>What just happened?</b><br>

`WHERE` narrows the result. Instead of all six patients, the query returns only the rows relating to one specific person. 

Notice that instead of `SELECT *`, the query specifies exactly two column names — `patient_name` and `date_of_birth`. This controls what information comes back; the `patient_id` and `nhs_number` columns are not returned because the query did not ask for them.

These two techniques — column selection and row filtering — are the building blocks of almost every real-world SQL query.
</div>

### Worked Example 3: Querying Across Tables with JOIN

**What you will see:** A query that combines data from all three tables, displaying appointment records with patient names and doctor names rather than numeric IDs.

In [ ]:
%%sql
SELECT
    a.appointment_date,
    a.appointment_time,
    p.patient_name,
    d.doctor_name,
    a.appointment_type,
    a.attended
FROM appointments a
JOIN patients p ON a.patient_id = p.patient_id
JOIN doctors  d ON a.doctor_id  = d.doctor_id
ORDER BY a.appointment_date, a.appointment_time;

<div class="alert alert-block alert-info">
<b>What just happened?</b><br>

`JOIN` connects tables using the foreign key relationships drawn in the ERD. The appointments table stores `patient_id` and `doctor_id`; `JOIN` looks up the actual names from the patients and doctors tables.

This is why normalisation works. The database stores data without redundancy — names appear once — and then reconstructs the full picture on demand when a query needs it. `ORDER BY` sorts the results by date and time, making the output readable as a schedule.

The `ON` clause specifies how the tables connect: `a.patient_id = p.patient_id` tells the database to match each appointment row to the patient row with the same ID. This is the ERD relationship made operational.
</div>

### Worked Example 4: Aggregation — Counting and Summarising

You will often want to summarise data rather than list every row. For example, how many appointments does each doctor have? The query below counts appointments per doctor and sorts the results to show the busiest doctors first.

In [ ]:
%%sql
SELECT
    d.doctor_name,
    COUNT(*) AS appointment_count
FROM appointments a
JOIN doctors d ON a.doctor_id = d.doctor_id
GROUP BY d.doctor_name
ORDER BY appointment_count DESC;

<div class="alert alert-block alert-info">
<b>What just happened?</b><br>

`COUNT(*)` counts rows. `GROUP BY` groups them — in this case, by doctor name. The result is one summary row per doctor, showing how many appointments each one has. `AS appointment_count` gives the counted column a readable name in the output.

This is how summary reports are generated from a database — 'How many appointments does each doctor have this week?' or 'Which appointment type is most common?' In a real practice management system, queries like this power the dashboards managers use every day.
</div>

<div class="alert alert-block alert-info">
<b>Further Information</b><br>

[Learn SQL — Interactive SQL Tutorial — SQLBolt (SQLBolt, no date)](https://sqlbolt.com/) — Browser-based interactive SQL lessons with immediate feedback. Work through Lessons 1–4 to practise the SELECT, WHERE, and filtering concepts you have seen in this Part. Lessons 6–7 cover multi-table queries with JOIN.

*Estimated time: 10–15 minutes for Lessons 1–4. Free, no login required.*
</div>

<div class="alert alert-block alert-success">
<b>Your turn — Task 1: Change a WHERE condition</b><br>

The query below finds all appointments for Dr Amara Okafor. Change the doctor's name to find appointments for a different doctor. Pick any doctor from the doctors table you saw above.

Change only the value inside the single quotes on the `WHERE` line — do not modify any other part of the query.

<details>
    <summary style="color:black;font-weight:bold">Click here to reveal the solution (example: Dr Rajesh Kumar)</summary>

```sql
SELECT
    a.appointment_date,
    a.appointment_time,
    p.patient_name,
    a.appointment_type
FROM appointments a
JOIN patients p ON a.patient_id = p.patient_id
JOIN doctors  d ON a.doctor_id  = d.doctor_id
WHERE d.doctor_name = 'Dr Rajesh Kumar'
ORDER BY a.appointment_date, a.appointment_time;
```

</details>
</div>

In [ ]:
%%sql
SELECT
    a.appointment_date,
    a.appointment_time,
    p.patient_name,
    a.appointment_type
FROM appointments a
JOIN patients p ON a.patient_id = p.patient_id
JOIN doctors  d ON a.doctor_id  = d.doctor_id
WHERE d.doctor_name = 'Dr Amara Okafor'  -- TODO: Change this to a different doctor's name
ORDER BY a.appointment_date, a.appointment_time;

<div class="alert alert-block alert-success">
<b>Your turn — Task 2: Change a date filter</b><br>

The query below finds all appointments on 6 January 2026. Change the date to find appointments on a different day. Look at the appointment dates in the output from Worked Example 3 above to pick a date that has appointments.

Change only the date value inside the single quotes — do not modify any other part of the query.

<details>
    <summary style="color:black;font-weight:bold">Click here to reveal the solution (example: 2026-01-07)</summary>

```sql
SELECT
    a.appointment_time,
    p.patient_name,
    d.doctor_name,
    a.appointment_type
FROM appointments a
JOIN patients p ON a.patient_id = p.patient_id
JOIN doctors  d ON a.doctor_id  = d.doctor_id
WHERE a.appointment_date = '2026-01-07'
ORDER BY a.appointment_time;
```

</details>
</div>

In [ ]:
%%sql
SELECT
    a.appointment_time,
    p.patient_name,
    d.doctor_name,
    a.appointment_type
FROM appointments a
JOIN patients p ON a.patient_id = p.patient_id
JOIN doctors  d ON a.doctor_id  = d.doctor_id
WHERE a.appointment_date = '2026-01-06'  -- TODO: Change this to a different date
ORDER BY a.appointment_time;

<div class="alert alert-block alert-success">
<b>Your turn — Task 3: Complete a SQL query (stretch)</b><br>

Complete the SQL query below to find all patients whose date of birth is before 1980. The structure is provided — fill in the three blanks.

**Hint:** Look at the column names in the patients table from Worked Example 1. You need the column that stores date of birth, and the table that contains patient records.

<details>
    <summary style="color:black;font-weight:bold">Click here to reveal the solution</summary>

```sql
SELECT patient_name, date_of_birth
FROM patients
WHERE date_of_birth < '1980-01-01'
ORDER BY date_of_birth;
```

**Why this works:** `date_of_birth` is stored as text in the format `YYYY-MM-DD`. Comparing text in this format works correctly because the year comes first — '1979' sorts before '1980' alphabetically, which also happens to be chronologically correct.

</details>
</div>

In [ ]:
%%sql
-- Complete the blanks to find patients born before 1980
SELECT ___, ___
FROM ___
WHERE date_of_birth < '1980-01-01'
ORDER BY date_of_birth;

---

## Part 4: Connecting Architecture to Your Workplace

### The Principles Apply Everywhere

The GP practice scenario in this notebook is one example. The design principles behind it are universal.

Every database you interact with was designed using the same ideas. An NHS trust's patient record system separates patient demographics, clinical episodes, and prescriptions into related tables — updating a patient's address in one place changes it everywhere. A retailer's inventory system links products, suppliers, and stock locations, allowing a query to tell you instantly which supplier provides a product that is running low. A council's planning application system connects applicants, properties, and case officers, generating workload reports from a single `COUNT` and `GROUP BY`.

The ERD you read in Part 1 is the same kind of diagram database architects draw before building any of these systems. The normalisation you saw in Part 2 is the same analysis those architects apply to prevent the data quality problems that make systems unreliable. The SQL queries in Part 3 are the same queries that power the dashboards and reports people use every day.

You may never write a `CREATE TABLE` statement in your role. But understanding that tables have relationships, that redundancy causes update anomalies, and that a `JOIN` reconstructs related information — that understanding changes how you read a database schema, discuss a system with a developer, or evaluate why a report is showing unexpected results.

---

## Reflection

Reflect on the following question and record your thoughts in the cell below. Ground your answer in an example from your workplace, using **approximately 100 words** in total.

Your answer here forms part of your Module Challenge Part A portfolio and is assessed against **MLO2:** *Critically reflect upon data management systems and their role in digital and technology solutions*.

Where you draw on external sources, cite them in Harvard style. This does not count towards your word count.

Once you commit your notebook, your Coach will be able to provide feedback on your response. You can develop your answer further before your final submission.

<div class="alert alert-block alert-info">

1. Using this week's normalisation principles, would you defend storing in a medical clinic's appointment datablase the doctor's name in every appointment row, the unnormalised approach? Should it be normalised? Justify your position in terms of what could go wrong, rather than just naming the normal form.

</div>

*Write your responses here.*

1. 

References (if applicable):

---

## Summary

**You have completed Week 2 Section 4.** You can now:

- Read the Entity Relationship Diagram for a GP practice database and implemented it in SQL, creating patients, doctors, and appointments tables linked by primary and foreign keys
- Compared an unnormalised `appointments_flat` table against three normalised tables, identifying the update, insertion, and deletion anomalies that normalisation prevents
- Modified SELECT, WHERE, and JOIN queries against the patients, doctors, and appointments tables, including filtering by doctor name and date, and counting appointments per doctor with GROUP BY
- Connected ERDs, normalisation, and SQL queries to database systems in your own workplace, such as patient records, inventories, or case management systems

**Key takeaways:**

- ERDs visualise database structure before any code is written: entities, attributes, and relationships, including cardinality
- Normalisation (1NF to 2NF to 3NF) reduces redundancy by splitting data into related tables, each responsible for one set of facts
- SQL queries (SELECT, WHERE, JOIN, GROUP BY) retrieve and combine data from normalised tables; the database reconstructs the full picture on demand
- The same design principles apply whether the database holds patient records, customer orders, or council service requests

**What comes next:**

- **Workshop 1** (Jupyter and SQL): hands-on SQL practice with the Pagila database. The query patterns from Part 3 apply directly.
- **Week 3 Section 1** introduces data quality: the six dimensions used to judge whether data can be trusted, building on the architecture foundations from this week.

**Action items:**

1. Save this notebook (Ctrl+S or Cmd+S)
2. Commit and push your work to your GitHub repository with the commit message `"Complete Week 2 Section 4 data architecture exploration"`
3. Make sure the repository shows your latest commit and that all files are up to date
4. Share a link to your repository with your Coach in the DT401.2 component on Aptem
5. Make sure your Coach has access to your repository so they can view your work and provide feedback
